In [18]:
import os
import joblib
import librosa
import numpy as np
import pandas as pd


In [19]:
def extract_features(filepath, sr=22050, duration=3.0):
    y, sr = librosa.load(filepath, sr=sr, duration=duration)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc_feat = np.hstack([mfcc.mean(axis=1), mfcc.std(axis=1)])

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_feat = np.hstack([chroma.mean(axis=1), chroma.std(axis=1)])

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    centroid_feat = np.array([centroid.mean(), centroid.std()])

    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_feat = np.array([zcr.mean(), zcr.std()])

    rms = librosa.feature.rms(y=y)
    rms_feat = np.array([rms.mean(), rms.std()])

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    rolloff_feat = np.array([rolloff.mean(), rolloff.std()])

    return np.hstack([
        mfcc_feat,
        chroma_feat,
        centroid_feat,
        zcr_feat,
        rms_feat,
        rolloff_feat
    ])


In [20]:
model = joblib.load('../models/rf_model.pkl')
le = joblib.load('../models/rf_label_encoder.pkl')


In [21]:
df = pd.read_csv('../data/dataset.csv')

AUDIO_FEATURES = [
    'valence','energy','tempo','danceability',
    'acousticness','instrumentalness','loudness'
]

df = df[['track_name','artists','track_genre'] + AUDIO_FEATURES].dropna()
df = df.drop_duplicates(subset=['track_name','artists']).reset_index(drop=True)


In [22]:
EMOTION_PROFILES = {
    'happy': {'targets': [0.85,0.80,0.72,0.82,0.20,0.08,0.72]},
    'sad': {'targets': [0.20,0.25,0.30,0.30,0.70,0.30,0.30]},
    'angry': {'targets': [0.30,0.95,0.86,0.58,0.10,0.15,0.90]},
    'fearful': {'targets': [0.25,0.55,0.50,0.35,0.45,0.35,0.45]},
    'neutral': {'targets': [0.50,0.50,0.50,0.50,0.40,0.25,0.50]},
    'calm': {'targets': [0.60,0.25,0.30,0.40,0.78,0.50,0.25]},
    'surprised': {'targets': [0.72,0.78,0.68,0.68,0.18,0.10,0.72]}
}


In [23]:
def predict_emotion(audio_path):
    feats = extract_features(audio_path).reshape(1, -1)
    pred = model.predict(feats)[0]
    return le.inverse_transform([pred])[0]


In [33]:
def recommend_songs(emotion, top_n=10):

    # 🎯 Get target vector for emotion
    target = np.array(EMOTION_PROFILES[emotion]["targets"])

    # 🔥 FILTER BAD DATA (THIS IS WHAT YOU ASKED)
    df_filtered = df[
    (df['valence'] > 0.3) &
    (df['energy'] > 0.4) &
    (~df['track_genre'].str.contains(
        'sleep|noise|study|comedy|kids|spoken|podcast',
        case=False, na=False
    ))
]


    scores = []

    # 🎵 Compute similarity
    for _, row in df_filtered.iterrows():
        song_vec = row[AUDIO_FEATURES].values.astype(float)

        score = 1 - np.mean(np.abs(song_vec - target))

        scores.append(score)

    df_copy = df_filtered.copy()
    df_copy["score"] = scores

    # 🎯 Return top songs
    return df_copy.sort_values("score", ascending=False).head(top_n)


In [34]:
def mood_music_pipeline(audio_path, top_n=10):

    emotion = predict_emotion(audio_path)

    print("Detected Emotion:", emotion)

    recs = recommend_songs(emotion, top_n)

    print(recs)

    return emotion, recs


In [35]:
test_file = '../data/archive/Actor_01/03-01-03-01-01-01-01.wav'
emotion, recs = mood_music_pipeline(test_file, top_n=10)


Detected Emotion: happy
                                              track_name  \
36768                                      Baila Cholita   
30259                         Go Power At Christmas Time   
35481                                     Life After You   
45668                                          In Dreams   
5117                                        Black Dragon   
7180                                   Love Is Blindness   
30169                                       Caixa Postal   
38734                    Ocean Prime (feat. Boldy James)   
56154                                 Casa Pré-Fabricada   
66355  Can't Help Falling In Love - Live at Mid-South...   

                         artists  track_genre  valence  energy   tempo  \
36768               William Luna       guitar    0.853   0.750  47.482   
30259                James Brown         funk    0.805   0.680  49.211   
35481                   Daughtry       grunge    0.309   0.764  51.316   
45668              